# Tutorial 03

Agenda:
- Descriptores Invariantes Locales
- Matching



# Setup

Este tutorial se puede ejecutar local en Jupyter lab o utilizar Google Colab.

## En Google Colab 
Este tutorial se provee junto con archivos de recursos dentro de un archivo ".zip".
En caso de ejecutar en Google Colab hay que:

1. Descomprimir el zip en algún lado
2. Subir el contenido del zip a Google Drive en alguna carpeta (por ejemplo `udesa/I308/tutoriales/tutorial_X`)
3. Abrir este notebook .ipynb 

In [ ]:
import os
import sys

# TODO: establecer el path en caso de trabajar con Colab
DRIVE_DIR = "udesa/I308/tutoriales/tutorial_03"

# detecta si estamos corriendo en Google Colab
try:
  from google.colab import drive
  COLAB = True
except:
  COLAB = False

if COLAB:
    # monta Google Drive
    drive.mount('/content/drive')

    base_path = "/content/drive/MyDrive/"
    path = os.path.join(base_path, DRIVE_DIR)
    
    %cd {path}
    sys.path.append(path)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# instalamos el paquete de utilidades
!pip install -qq git+https://github.com/udesa-vision/i308-utils.git

from i308_utils import imshow, show_images

# Motivación

Tengo un objeto y tengo una escena con objetos.

¿Cómo podemos armar un método que encuentre el objeto en la escena?

In [ ]:
import cv2
import numpy as np

def resize(img, w):
    w_, h_ = img.shape[1], img.shape[0]
    h = int(h_ * w / w_)
    return cv2.resize(img, (w, h))
    

objeto = cv2.imread('res/object_petrona.jpeg', cv2.IMREAD_GRAYSCALE) # queryImage
escena = cv2.imread('res/scene13.jpeg', cv2.IMREAD_GRAYSCALE) # trainImage


img_obj = resize(objeto, 640)
img_obj.shape


show_images([
    objeto, escena
], ["objeto", "escena"])


# Features en Imágenes

## Armando el rompecabezas

Armar un rompecabezas requiere ensamblar piezas pequeñas de una imagen.

¿Cómo hacemos (nosotros humanos) para hacer esta tarea?

=> buscamos patrones específicos o características específicas que sean únicas, que puedan rastrearse y compararse fácilmente.

Miremos la siguiente imagen: con los 6 parches A..F.

<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_03/feature_building.jpg?raw=true" width="60%"/>

¿Para cuáles parches podemos encontrar su ubicación exacta en la imagen original?

A y B: difícil.
C y D: mejor pero aun no se puede.
E y F: podemos encontrar una posicion unívoca. :)

¿Qué características tienen los parches E y F?

=> **los corners o esquinas son buenos features para encontrar cosas en las imagenes**.

Ok pero... 

¿Cómo podemos detectar los corners por código?


# Esquinas o Corners

Un corner es un cambio rápido de dirección en una curva. Los corners son efectivos para hacer coincidir cosas similares.

Una manera de encontrar corners es armar una métrica en el parche que mida cambios de intensidad grandes en varias dimensiones.

De esa manera funcionan la familia de detectores de corners moravec:

- Harris
- Shi-Tomasi


# Detectores de Corners

Si nos paramos en un pixel (x, y) de la imagen y miramos cómo varía la intensidad en un pequeño parche (u, v)
¿cómo cambiaría la intensidad si (x, y) es una esquina?


La suma de diferencias al cuadrado nos da una métrica que captura la variacion de intensidad a medida que nos desplazamos en el parche:

$ E(u, v) = \sum\limits_{(x, y) \in patch} w(x, y) . [ I(x+ u, y + v) - I( x, y ) ]^2$


Usando el desarrollo de Taylor en 2 variables se puede aproximar:

$
\begin{align}
    E(u, v) ≈
      \begin{bmatrix}
           u &
           v
      \end{bmatrix}
      M   
      \begin{bmatrix}
           u \\
           v \\
      \end{bmatrix}
  \end{align}
$

En donde:

$
M = \begin{bmatrix}
      S_{xx}   & S_{xy} \\
      S_{xy} & S_{yy}
  \end{bmatrix}
$

- $ S_{xx} = G ⋆ I_x ^2 $

- $ S_{yy} = G ⋆ I_y ^2 $

- $ S_{xy} = G ⋆ I_x * I_y $

$G$ es un filtro gaussiano, que suaviza quita ruido y da estabilidad al algoritmo.

$I_x$, $I_y$, son las derivadas de primer orden de la imagen, que podemos obtener usando Sobel, por ejemplo.

## Procedimiento:

1. Armar la matriz M
2. Calcular R, la respuesta de esquinas, en función de los autovalores de M: $λ_1$ y $λ_2$.
3. Aplicar thresholding y supresión de no-máximos

### Harris

Utiliza:

$ R = det(M) - k . tr(M)^2 $

en donde:

- $det(M) = λ_1 . λ_2 = S_{xx} . S_{yy} - {S_{xy}}^2 $
- $tr(M) = λ_1 + λ_2 = S_{xx} + S_{yy} $

- $k$ es un valor de sensibilidad entre 0.04 y 0.06.

### Shi-Tomasi

Utiliza:

$ R = min(λ_1, λ_2) $


# Método de Harris

Implementemos en python una función que encuentre corners usando el método de harris

In [ ]:
img = cv2.imread("res/icosahedron.jpg")

imshow(img)

In [ ]:
import cv2
import numpy as np


def harris(gray, k=0.04, sigma=1, ksize=3):

    # calcula el gradiente
    dx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=ksize)
    dy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=ksize)

    # calcula los elementos de M
    dx2 = dx ** 2
    dy2 = dy ** 2
    dxy = dx * dy

    # suaviza usando gaussian kernel
    g = cv2.getGaussianKernel(sigma=sigma, ksize=ksize, ktype=cv2.CV_64F)
    sxx = cv2.sepFilter2D(dx2, cv2.CV_64F, g, g)
    syy = cv2.sepFilter2D(dy2, cv2.CV_64F, g, g)
    sxy = cv2.sepFilter2D(dxy, cv2.CV_64F, g, g)
    
    # calcula la respuesta usando Harris
    det_M = (sxx * syy) - (sxy ** 2)
    trace_M = sxx + syy
    R = det_M - k * (trace_M ** 2)

    # normaliza la respuesta
    R = (R - R.min()) / (R.max() - R.min())

    return R


gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
R = harris(gray)

show_images([
    gray,
    R
])

In [ ]:
from matplotlib import pyplot as plt

thresh = 0.3
x, y = np.where(R > thresh)

print("corners encontrados:", len(x))

f, ax = imshow(gray, show=False)
ax.scatter(y, x, color='red', marker='x')
plt.show()



## Harris en OpenCV

OpenCV provee la función [`cv2.cornerHarris`](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html#gac1fc3598018010880e370e2f709b4345) que calcula la imagen con la respuesta de detección de corners.

In [ ]:
# TODO
# Hallar la imagen de respuesta de corners y graficarlos
# usando cv2.cornerHarris

img = cv2.imread("res/icosahedron.jpg")
img = cv2.imread("res/blox.jpg")
# img = cv2.imread("recursos/checkerboard.jpg")

import numpy as np

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.title("original")
plt.imshow(rgb)
plt.show()



In [ ]:
gray = np.float32(gray)
R = cv2.cornerHarris(gray, 2, 3, 0.04)
R = cv2.normalize(R, None, 0, 1, cv2.NORM_MINMAX)

plt.imshow(R, cmap='gray')
plt.title("response")
plt.show()



In [ ]:
# threshold
thresh = 1.01 * np.median(R)

x, y = np.where(R > thresh)
plt.imshow(R, cmap='gray')
plt.scatter(y, x, s=1, color='lightgreen')
plt.title("threshold")
plt.show()


In [ ]:
# threshold y nms
radius = 5
#thresh = 1.05 * np.median(R)

# cómo funciona este método de NMS?
# que problemas puedo encontrar?
# cómo podría solucionarlos?

sze = 2 * radius + 1
kernel = np.ones((sze, sze), dtype=np.uint8)
mx = cv2.dilate(R, kernel)
x, y = np.where((R == mx) & (R > thresh))

plt.imshow(mx, cmap='gray')
plt.title("dilated")
plt.show()

plt.imshow(rgb)
plt.scatter(y, x, s=50, marker='+', color='red')
plt.title("corners")
plt.show()

# GFFT (shi-tomasi)

OpenCV provee la funcion [`goodFeaturesToTrack()`](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html#ga1d6bb77486c8f92d79c8793ad995d541) que nos permite encontrar corners.



In [ ]:
# TODO:
# - usando cv2.goodFeaturesToTrack() hallar corners en la imagen y graficarlos.
# - Comparar los métodos Harris y Shi-tomasi
# - Qué diferencia tiene esta funcion con cv2.cornerHarris ?

def find_corners(img, method='harris'):

  useHarrisDetector = method=='harris'

  img = np.float32(img)
  corners = cv2.goodFeaturesToTrack(
      img,
      maxCorners=1000,
      qualityLevel=0.05,
      minDistance=11,
      useHarrisDetector=useHarrisDetector
  )

  corners = corners.reshape(-1, 2)
  return corners


def plot_corners(img, **kwargs):

  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  corners = find_corners(gray, **kwargs)

  plt.imshow(gray, cmap='gray')
  plt.scatter(corners[:, 0], corners[:, 1], s=50, marker='+', color='red')
  plt.show()


In [ ]:
plot_corners(img, method='harris')
plot_corners(img, method='shi-tomasi')


# SIFT

Los corners son buenos features en situaciones particulares. 
Pero para ciertas aplicaciones se quedan un poco cortos.

¿Qué pasa cuando escalamos la imagen?

Una esquina podría no ser esquina si se escala la imagen: Harris no es invariante a cambios de escala.

<img src="https://raw.githubusercontent.com/udesa-vision/i308-resources/refs/heads/main/tutoriales/tutorial_03/sift/scale_problem.webp" />

En 2004, David Lowe (British Columbia), presentó un nuevo método, Scale Invariant Feature Transform (SIFT) en su paper, [Distinctive Image Features from Scale-Invariant Keypoints](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf), que extrae keypoints y calcula descriptores.


Algoritmo de SIFT:

**1. Detección de Keypoints en espacio y escala:**
Se detectan keypoints (puntos clave que sean invariantes con la escala y la orientación). Para eso se construye una representación espacial a escala de la imagen aplicando kernels gausianos variando sigma. Se aproximan LoG mediante DoG. Los keypoints son máximos y mínimos locales en las imágenes DoG.


<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_03/sift/SIFT1.png?raw=true" width="60%"/>

<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_03/sift/SIFT2.png?raw=true"/>

**2. Localizacion de Keypoints**
Los keypoints detectados se refinan a precisión de subpíxeles ajustando una función cuadrática en 3D e interpolando. Luego se descartan keypoints con bajo contraste y los keypoints en edges.


**3. Asignación de orientación:** 
Se calcula la magnitud y orientación del gradiente alrededor del keypoint.
Se arma una ventana circular alrededor del keypoint candidato. El radio es proporcional a la escala. Se computan los gradientes de la imagen (sobre la imagen suavizada en la pirámide en la escala correspondiente). Se arma un histograma de 36 bins de orientaciones (cada bin cubre 10º), en donde se acumulan las orientaciones del gradiente en el parche ponderando con una gaussiana centrada en el keypoint.

**4. Se calcula un descriptor para cada keypoint.**

para una parche calculando histogramas de gradiente en subregiones (generalmente subregiones de 4x4).
La información de gradiente de cada subregión se agrupa en 8 orientaciones, lo que da como resultado un vector de 128 dimensiones (4x4x8) para keypoint.

<img src="https://github.com/udesa-vision/i308-resources/blob/main/tutoriales/tutorial_03/sift/SIFT4.jpg?raw=true"
    width="100%" />

## SIFT - Escala 

Cómo resuelve SIFT el problema de la escala?



Función Gausseana 1D:

$ g(x) = \frac{1}{\sqrt{2 . π σ^2}} . e^{- \frac{x^2}{2 . σ^2}} $


Derivada primera:

$ g'(x) = \frac{-x}{σ^2} . g(x)  $


Derivada segunda:

$ g''(x) = \frac{x^2 - σ^2}{σ^4} . g(x)  $


In [ ]:
import numpy as np

def gaussian_kernel_1d(sigma=1, ksize=None, K=None):

  if ksize is None:
    # if ksize is None, ksize se
    # computará automáticamente con un tamaño de 2 * sigma + 1
    ksize = int(2 * sigma + 1)

  a = ksize // 2

  c = 2 * sigma**2
  if K is None:
    K = 1 / np.sqrt(c * np.pi)
  t = np.linspace(-a, a, ksize)
  w = K * np.exp(-(t**2) / c)

  w = w / w.sum()
  return w

def gaussian_kernel_first_derivative(sigma=1, ksize=None):
    if ksize is None:
        ksize = int(2 * sigma + 1)

    # Create the Gaussian kernel
    gaussian_kernel = gaussian_kernel_1d(sigma, ksize)

    # Compute the center index of the kernel
    center = ksize // 2

    # Compute the first derivative of the Gaussian kernel
    x = np.linspace(-center, center, ksize)
    gaussian_derivative = -x / (sigma**2) * gaussian_kernel

    # Normalize the derivative kernel
    gaussian_derivative /= np.sum(np.abs(gaussian_derivative))

    return x, gaussian_derivative


def gaussian_kernel_second_derivative(sigma=1, ksize=None):

    if ksize is None:
        ksize = int(2 * sigma + 1)

    # Create the Gaussian kernel
    gaussian_kernel = gaussian_kernel_1d(sigma, ksize)

    # Compute the center index of the kernel
    center = ksize // 2

    # Compute the second derivative of the Gaussian kernel
    x = np.linspace(-center, center, ksize)
    gaussian_second_derivative = (x**2 - sigma**2) / (sigma**4) * gaussian_kernel

    # Normalize the second derivative kernel
    gaussian_second_derivative /= np.sum(np.abs(gaussian_second_derivative))

    return x, gaussian_second_derivative

### Repaso detección de bordes

¿Cómo encontrabamos los bordes de una imagen?

- Derivadas primeras ej. Sobel
- Derivadas segundas Laplaciano

Problema, si hay ruido detectamos el ruido como borde.

¿Entonces qué hacíamos?

=> primero suavizabamos y luego pasabamos el filtro.

Pero como vimos, las convoluciones cumplen la propiedad asociativa. Por lo que si suavizamos y luego derivamos, es teóricamente equivalente a derivar el suavizado y luego aplicar:

- $ E1 = (I ★ G) ★ S = I ★ (G ★ S) = I ★ S_x $

- $ E2 = (I ★ G) ★ L = I ★ (G ★ L) = I ★ LoG $

¿Qué pinta tienen los kernels $S_x$ y $LoG$?

### Blob Detection

usando LoG y DoG

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Creamos una señal sintética en 1D con un borde y ruido gausiano
mx = 2000
np.random.seed(0)
x = np.linspace(0, mx, mx)
signal = np.where(x < mx / 2, 0, 1) + np.random.normal(0, 0.05, x.shape)


ksize = 256
# Computo un kernel gausiano con la primera derivada
dx, g_prime = gaussian_kernel_first_derivative(sigma=20, ksize=ksize)
# Computo un kernel gausiano con la segunda derivada (LoG)
dx, log = gaussian_kernel_second_derivative(sigma=20, ksize=ksize)


# Aplico los filtros
edges1 = cv2.filter2D(signal, -1, -g_prime)
edges2 = cv2.filter2D(signal, -1, log)


# Grafico los resultados:
plt.figure(figsize=(12, 8))

plt.subplot(5, 1, 1)
plt.title('Image / Signal')
plt.plot(x, signal, label='Image', color='blue')
plt.gca().set_xlim(0, mx)
plt.legend()
plt.grid()

plt.subplot(5, 1, 2)
plt.title('Gaussian First Derivative (Sx)')
plt.plot(dx + mx/2, g_prime, label='Gauss 1st', color='red')
plt.gca().set_xlim(0, mx)
plt.legend()
plt.grid()


plt.subplot(5, 1, 3)
plt.title('Edge response using 1st derivative')
plt.plot(x, edges1, label='Edge response', color='green')
plt.gca().set_xlim(0, mx)
plt.legend()
plt.grid()


plt.subplot(5, 1, 4)
plt.title('Gaussian Second Derivative (LoG)')
plt.plot(dx + mx/2, log, label='LoG', color='red')
plt.gca().set_xlim(0, mx)
plt.legend()
plt.grid()


plt.subplot(5, 1, 5)
plt.title('Edge response using LoG')
plt.plot(x, edges2, label='Edge response', color='green')
plt.gca().set_xlim(0, mx)
plt.legend()
plt.grid()


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def show_log_response(pulse_size=20):

  # Creamos una señal sintética en 1D con un borde y ruido gausiano
  mx = 2000
  np.random.seed(0)
  x = np.linspace(0, mx, mx)
  signal = np.where(
      np.abs(x - (mx / 2)) > pulse_size,
      0, 1
  ) + np.random.normal(0, 0.05, x.shape)


  ksize = 256

  # Computo un kernel gausiano con la segunda derivada (LoG)
  dx, log = gaussian_kernel_second_derivative(sigma=20, ksize=ksize)


  # Aplico los filtros
  edges1 = cv2.filter2D(signal, -1, log)


  # Grafico los resultados:
  plt.figure(figsize=(12, 8))

  plt.subplot(5, 1, 1)
  plt.title('Image / Signal')
  plt.plot(x, signal, label='Image', color='blue')
  plt.gca().set_xlim(0, mx)
  plt.legend()
  plt.grid()

  plt.subplot(5, 1, 2)
  plt.title('LoG')
  plt.plot(dx + mx/2, log, label='LoG', color='red')
  plt.gca().set_xlim(0, mx)
  plt.legend()
  plt.grid()


  plt.subplot(5, 1, 3)
  plt.title('LoG response')
  plt.plot(x, edges1, label='LoG response', color='green')
  plt.gca().set_xlim(0, mx)
  plt.legend()
  plt.grid()

  plt.tight_layout()
  plt.show()

sizes = [100, 50, 20]
for size in sizes:
  print(f"blob size: {size}")
  show_log_response(pulse_size=size)



### Laplacian of Gaussian en 2D

LoG en 2d:

$ LoG(x, y) = -\frac{1}{π σ^4} (1-\frac{x^2+y^2}{2σ^2}) . e^{-\frac{x^2 + y^2}{2σ^2}}$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Define the function to create a 2D LoG filter
def laplacian_of_gaussian(size, sigma):
    # Create a coordinate grid
    ax = np.linspace(-size // 2 + 1., size // 2 + 1., 1024)
    xx, yy = np.meshgrid(ax, ax)

    # Compute the Laplacian of Gaussian filter
    norm = (xx**2 + yy**2) / (2 * sigma**2)
    log_filter = - (1 / (np.pi * sigma**4)) * (1 - norm) * np.exp(-norm)

    return xx, yy, log_filter

# Parameters for the LoG filter
size = 100   # Size of the filter
sigma = 10.0 # Standard deviation of the Gaussian

# Create the LoG filter
xx, yy, log_filter = laplacian_of_gaussian(size, sigma)

# Plot the LoG filter as a 3D surface
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(xx, yy, log_filter, cmap='inferno')

# Set plot labels and title
ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('Filter value')

ax.set_title('Laplacian of Gaussian (LoG) Filter')

plt.show()

### DoG

Calcular LoG es una fórmula pesada, mejor aproximamos LoG con DoG. Diferencia de gausianas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Funcion Gausiana 1D
def gaussian(x, sigma):
    return np.exp(-x**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))

# LoG 1D
def laplacian_of_gaussian(x, sigma):
    norm = x**2 / (2 * sigma**2)
    log = - (1 / (np.pi * sigma**4)) * (1 - norm) * np.exp(-norm)
    return log

# DoG 1D
def difference_of_gaussians(x, sigma1, sigma2):
    gauss1 = gaussian(x, sigma1)
    gauss2 = gaussian(x, sigma2)
    return gauss2 - gauss1


# defino parámetros
x = np.linspace(-10, 10, 1000)
sigma = 1.0
sigma1 = 1.0
sigma2 = 2.0


# Computa LoG y DoG
log = laplacian_of_gaussian(x, sigma)
dog = difference_of_gaussians(x, sigma1, sigma2)

# Ploteo resultados
plt.figure(figsize=(10, 6))

plt.plot(x, log, label='Laplacian of Gaussian (LoG)', color='blue')
plt.plot(x, dog, label='Difference of Gaussians (DoG)', color='red', linestyle='--')

plt.title('Aproximación of LoG con DoG')
plt.xlabel('x')
plt.ylabel('amplitud')
plt.legend()
plt.grid(True)
plt.show()

### Buscando Blobs con DoG

Variando sigma podemos detectar blobs de distintas escalas:

Aplicando un sigma mas grande detectamos objetos a mayor escala. Y con un sigma mas pequeño objetos mas pequeños ej detalles.


In [ ]:
# Read the image
img = cv2.imread("res/lenna.png", cv2.IMREAD_GRAYSCALE)  # Convert to grayscale

def compute_dog(img, sigma1, sigma2):

  # Apply Gaussian blurring with different sigma values
  gauss1 = cv2.GaussianBlur(img, (0, 0), sigma1)
  gauss2 = cv2.GaussianBlur(img, (0, 0), sigma2)

  # Compute the Difference of Gaussians (DoG)
  dog = gauss1 - gauss2

  return dog


# Define sigma values
sigma1 = 1.0
sigma2 = 2.0

# Plot the results
plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
plt.title('Original Image')
plt.imshow(img, cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title(f'DoG (sigma={1} and {2})')
plt.imshow(compute_dog(img, 1.0, 2.0), cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title(f'DoG (sigma={10} and {20})')
plt.imshow(compute_dog(img, 10.0, 20.0), cmap='gray')
plt.axis('off')


plt.show()

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


# Crea una imagen sintética con blobs de distintos tamaños
def create_synthetic_image(size, blobs_info):
    image = np.zeros(size, dtype=np.float32)
    for x, y, radius in blobs_info:
        cv2.circle(image, (x, y), radius, color=1, thickness=-1)
    return image

# Dada una imagen y dos sigmas calcula DoG
def compute_DoG(image, sigma1, sigma2):
    gauss1 = cv2.GaussianBlur(image, (0, 0), sigma1)
    gauss2 = cv2.GaussianBlur(image, (0, 0), sigma2)
    DoG = gauss1 - gauss2
    return DoG


def non_maximum_suppression(image):

    # mantiene solamente maximos locales
    nms_image = image == np.max(image)

    return nms_image


# Parámetros de la imagen sintética:
size = (256, 256)
params = [
    (60, 60, 7),    # Blob 1: radio 7
    (125, 125, 15),  # Blob 2: radio 15
    (200, 200, 30)   # Blob 3: radio 30
]

# Crea la imagen sintética
image = create_synthetic_image(size, params)


# Grafica la imagen con blobs
plt.title('Imagen original con blobs')
plt.imshow(image, cmap='gray')
plt.axis('off')
plt.show()

# Computa DoG para diferentes escalas
sigma_base = [7, 15, 30]
epsilon = 0.01  # pequeña variacion alrededor del sigma base

# threshold = 0.005

# Prepare output image
output = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)  # Convert to BGR for color circles

# Define los colores distintos para cada circulo a graficar en el output
colors = [(1, 0, 0), (0, 1, 0), (0, 0, 1)]  # Red, Green, Blue
# colors = ['r', 'g', 'b']  # Red, Green, Blue

for sigma, color in zip(sigma_base, colors):
    dog = compute_DoG(image, sigma - epsilon, sigma + epsilon)

    plt.figure()
    plt.title(f'Difference of Gaussians (sigma={sigma - epsilon}, {sigma + epsilon})')
    plt.imshow(dog)
    plt.colorbar()
    plt.show()

    nms = non_maximum_suppression(dog)

    # Encuentra las coordenadas de los blobs
    y_coords, x_coords = np.where(nms > 0)

    for (x, y) in zip(x_coords, y_coords):
        # print(sigma, color)
        cv2.circle(output, (x, y), int(sigma), color=color, thickness=2)

# Plot the results
plt.title('Blobs detectados')
plt.imshow(output)
plt.axis('off')
plt.show()


## SIFT - Descartando keypoints malos

SIFT utiliza un método muy parecido al de Harris para descartar candidatos malos (ej, que caen en un borde y por lo tanto no tienen una ubicación robusta)

$
H = \begin{bmatrix}
      D_{xx}   & D_{xy} \\
      D_{xy} & D_{yy}
  \end{bmatrix}
$

El Hessiano se construye a partir de diferencias finitas en las imágenes de DoG.

Criterio:
Se usa un threshold radio $r$, si el cociente da menor que esta cota se mantiene el keypoint, si no se descarta.

$ \frac{ Tr^2 (H)}{Det(H)} = \frac{(λ_1 + λ_2)^2}{λ_1 . λ_2} < \frac{(r + 1)^2}{r} $


## SIFT - Cálculo de orientación dominante

¿Cómo podemos calcular la orientación dominante de un parche alrededor de un keypoint?

(lo vimos la vez pasada)

- $ | ∇I | = \sqrt{ S_x^2 + S_y^2 } $

- $ arctan( \frac{ S_y }{ S_x } ) $

En donde $S$ es la imagen suavizada.

## SIFT - Cálculo de descriptor

- Parche: se considera un parche en la vecindad del keypoint, girado para que quede alineado con la orientación dominante.

- Se divide el parche en subregiones de 4x4 (cada subregión es un pequeño bloque cuadrado de píxeles).

- Histogramas de gradiente para subregiones: para cada subregión, SIFT calcula un histograma de orientación local con 8 contenedores, que es un HoG pequeño. Cada píxel de la subregión contribuye al histograma en función de su magnitud y orientación del gradiente (en relación con la orientación del keypoint). Las contribuciones están ponderadas por una ventana gaussiana para dar más énfasis a los gradientes más cercanos al keypoint.

En 2004, David Lowe (British Columbia), presentó un nuevo método, Scale Invariant Feature Transform (SIFT) en su paper, [Distinctive Image Features from Scale-Invariant Keypoints](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf), que extrae keypoints y calcula descriptores.


Algoritmo de SIFT:

1. Detección de Keypoints en espacio y escala:
Se detectan keypoints (puntos clave que sean invariantes con la escala y la orientación). Para eso se construye una representación espacial a escala de la imagen aplicando kernels gausianos variando sigma. Se aproximan LoG mediante DoG. Los keypoints son máximos y mínimos locales en las imágenes DoG.

2. Localizacion de Keypoints
Los keypoints detectados se refinan a precisión de subpíxeles ajustando una función cuadrática en 3D e interpolando. Luego se descartan keypoints con bajo contraste y los keypoints en edges.

3. Asignación de orientación: Se calcula la magnitud y orientación del gradiente alrededor del keypoint.

4. Se calcula un descriptor para cada keypoint,
para una parche calculando histogramas de gradiente en subregiones (generalmente subregiones de 4x4).
La información de gradiente de cada subregión se agrupa en 8 orientaciones, lo que da como resultado un vector de 128 dimensiones (4x4x8) para keypoint.


# SIFT en OpenCV

OpenCV provee el algoritmo [SIFT](https://docs.opencv.org/4.x/d7/d60/classcv_1_1SIFT.html#a4264f700a8133074fb477e30d9beb331) que computa todo esto por nosotros.

# Volviendo al problema del principio...

Quiero encontrar el objeto en la imagen.

Está si o no?
Si está, dónde está?
Cual es la pose del objeto?


In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt


img_obj = cv.imread('res/object_petrona.jpeg', cv.IMREAD_GRAYSCALE) # queryImage
# img_obj = cv.imread('res/object_off.jpeg', cv.IMREAD_GRAYSCALE) # queryImage
# img_obj = cv.imread('res/object_control.jpeg', cv.IMREAD_GRAYSCALE) # queryImage
# img_obj = cv.imread('res/object_gorra.jpeg', cv.IMREAD_GRAYSCALE) # queryImage
# img_obj = cv.imread("res/object_termo.jpeg", cv.IMREAD_GRAYSCALE)

img_scn = cv.imread('res/scene13.jpeg', cv.IMREAD_GRAYSCALE) # trainImage
# img_scn = cv.imread('res/scene1.jpeg', cv.IMREAD_GRAYSCALE) # trainImage
# img_scn = cv.imread('res/scene2.jpeg', cv.IMREAD_GRAYSCALE) # trainImage
# img_scn = cv.imread('res/scene3.jpeg', cv.IMREAD_GRAYSCALE) # trainImage
# img_scn = cv.imread('res/scene9.jpeg', cv.IMREAD_GRAYSCALE) # trainImage

img_obj = resize(img_obj, 640)
img_obj.shape

show_images([
    img_obj, img_scn
], ["objeto", "escena"])


In [ ]:
# Usemos el algoritmo SIFT

algo = cv2.SIFT_create()

# algo = cv2.SIFT_create(
#    nfeatures=1000, sigma=1.5, nOctaveLayers=4
# )
# algo = cv.AKAZE_create() # (nfeatures=2000)
# algo = cv.SIFT_create() # (nfeatures=2000)


# find the keypoints and descriptors
kp_obj, des_obj = algo.detectAndCompute(img_obj, None)
kp_scn, des_scn = algo.detectAndCompute(img_scn, None)

In [ ]:
print("kps objeto", len(kp_obj), "descriptores objeto", len(des_obj))
print("kps escena", len(kp_scn), "descriptores escena", len(des_scn))

In [ ]:
keypoint_0 = kp_obj[0]

print("x, y:", keypoint_0.pt)
print("size:", keypoint_0.size)
print("octave:", keypoint_0.octave)
print("angle:", keypoint_0.angle)
print("response:", keypoint_0.response)


In [ ]:
descriptor_0 = des_obj[0]
descriptor_0

In [ ]:
kp_obj_image = cv2.cvtColor(img_obj, cv2.COLOR_GRAY2BGR)
kp_scn_image = cv2.cvtColor(img_scn, cv2.COLOR_GRAY2BGR)

kp_obj_image = cv2.drawKeypoints(
    kp_obj_image,
    kp_obj,
    0, (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
# Dibujo en rojo el primer keypoint
kp_obj_image = cv2.drawKeypoints(
    kp_obj_image,
    [keypoint_0],
    0, (0, 0, 255),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

kp_scn_image = cv2.drawKeypoints(
    kp_scn_image,
    kp_scn,
    0, (255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)


show_images([
    kp_obj_image, kp_scn_image
], ["keypoints objeto", "keypoints escena"])

# Matcheando Descriptores

Tenemos los descriptores del objeto y los descriptores de la escena.

Los descriptores de las mismas partes del objeto en la escena deberían parecerse.

Si tenemos dos descriptores $d_{obj}$ y $d_{scene}$ que se corresponden a lo mismo esperaríamos que:

$ distance( d_{obj}, d_{escena} ) < th $

Podemos hacer un for loop que para cada descriptor del objeto busque el descriptor de la imagen mas cercano (método de fuerza bruta).


In [ ]:
# Usamos fuerza bruta, 
candidate_matches = []

# itero por todos los descriptores del objeto
for i, d_obj in enumerate(des_obj):
    
    # computo la distancia contra los descriptores de la imagen
    distances = np.linalg.norm(des_scn - d_obj, axis=1)
    
    # busquemos el de minima distancia
    min_index = np.argmin(distances)
    
    # Creamos un objeto DMatch 
    match = cv2.DMatch(
        _queryIdx=i, 
        _trainIdx=min_index, 
        _distance=distances[min_index]
    )
    candidate_matches.append(match)



In [ ]:
# cross check?
# Ejemplo:
# Sup tengo imagenes A, B con los descriptores
# A1=1, A2=4
# B1=3, B2=5
# 
# NN(A1) = B1, pero NN(B1)= A2

In [ ]:
# Cross-check: 
# nos aseguramos que el mejor match es mutuo
matches = []

for match in candidate_matches:
    i = match.queryIdx
    min_index = match.trainIdx
    
    # hacemos el match inverso
    # buscando el descriptor más parecido en los descriptores del objeto
    reverse_distances = np.linalg.norm(des_obj - des_scn[min_index], axis=1)
    reverse_min_index = np.argmin(reverse_distances)
    
    # Si el indice reverso apunto al mismo descriptor, entonces es más confiable.
    if reverse_min_index == i:
        
        matches.append(match)

In [ ]:
# Ordena los matches por distancia
matches = sorted(matches, key=lambda x: x.distance)

# Nos quedamos con los 100 mejores matches
matches = matches[:100]

res_img = cv.drawMatches(
    img_obj,
    kp_obj,
    img_scn,
    kp_scn,
    matches,
    None,
    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()

In [ ]:
# Usando Fuerza bruta de CV!

# BFMatcher
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches = bf.match(des_obj, des_scn)

# Ordena los matches por distancia
matches = sorted(matches, key=lambda x: x.distance)

# Nos quedamos con los 100 mejores matches
matches = matches[:100]

res_img = cv.drawMatches(
    img_obj,
    kp_obj,
    img_scn,
    kp_scn,
    matches,
    None,
    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)
plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()


In [ ]:
# Usando Fuerza bruta, mejor filtrado.
# Mejoramos los matches que seleccionamos usando Lowe ratio test: 
# Consideramos los dos vecinos más cercanos, si la distancia al primero es mucho menor que la 
# distancia al segundo entonces es un feature bien característico. 
# Si no, por las dudas lo descartamos.

# BFMatcher
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
matches = bf.knnMatch(des_obj, des_scn, k=2)

# Nos quedamos con matches cuyo segundo mejor match está lejos
matchesMask = [[0, 0] for i in range(len(matches))]
for i, (m, n) in enumerate(matches):
    if m.distance < 0.5 * n.distance:
        matchesMask[i] = [1, 0]

# Dibuja sólo buenos matches
draw_params = dict(matchColor=(0, 255, 0),
                   singlePointColor=(255, 0, 0),
                   matchesMask=matchesMask,
                   flags=cv2.DrawMatchesFlags_DEFAULT)


res_img = cv2.drawMatchesKnn(
    img_obj, kp_obj, img_scn, kp_scn, matches, None, **draw_params
)

plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()


# FLANN: Fast Library for Approximate Nearest Neighbor

la solución exacta de matchear descriptores es muy cara. Requiere comparar todos contra todos.

Por eso OpenCV provee métodos para resolver el matching de manera mucho más rápida.
Si bien es una solución aproximada ANN, para lo que queremos resolver alcanza.



In [ ]:
# FLANN mejor!
# Flann acelera el matching de desciptores usando estructuras de datos.

# BFMatcher
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)  # or pass empty dictionary

flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des_obj, des_scn, k=2)

# Nos quedamos con matches cuyo segundo mejor match está lejos
matchesMask = [[0, 0] for i in range(len(matches))]
for i, (m, n) in enumerate(matches):
    if m.distance < 0.5 * n.distance:
        matchesMask[i] = [1, 0]

# Dibuja sólo buenos matches
draw_params = dict(matchColor=(0, 255, 0),
                   singlePointColor=(255, 0, 0),
                   matchesMask=matchesMask,
                   flags=cv2.DrawMatchesFlags_DEFAULT)


res_img = cv2.drawMatchesKnn(
    img_obj, kp_obj, img_scn, kp_scn, matches, None, **draw_params
)

plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()

# Otros métodos

Estos métodos están implementados en OpenCV.

- SIFT
- [SURF](https://docs.opencv.org/4.x/df/dd2/tutorial_py_surf_intro.html)
- [AKAZE](https://docs.opencv.org/4.x/db/d70/tutorial_akaze_matching.html)
- [ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html)


SIFT, SURF, AKAZE son descriptores numéricos.

ORB (basado en FAST y BRISK, es binario)

**NOTA**
SURF no va a andar si no instalan el módulo contrib y recompilan OpenCV con el flag OPENCV_ENABLE_NONFREE.


# Descriptores Binarios ORB

SIFT, SURF son features que funcionan muy bien, pero el cómputo es caro.
Si queremos **real time**, necesitamos acelerar las cosas. 

Eso motiva la existencia de descriptores binarios que realizan tests muy rápidos de intensidad de color.
como FAST, y BRIEF. La combinación de estos dos métodos dan lugar al método ORB invariante a rotación y a escala.  No tan preciso como SIFT pero mucho más rápido.


**Hamming Distance**
Los descriptores de los métodos binarios son Strings con ceros y unos, los descriptores no se pueden comparar con una distancia euclidea. En cambio vamos a utilizar distancia de hamming, que mide la cantidad de elementos diferentes en el string.


In [ ]:
algo = cv2.ORB_create()

# find the keypoints and descriptors
kp_obj, des_obj = algo.detectAndCompute(img_obj, None)
kp_scn, des_scn = algo.detectAndCompute(img_scn, None)


# Binary
bf = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=True)

# BFMatcher with default params
#bf = cv.BFMatcher()
matches = bf.match(des_obj, des_scn)

# Apply ratio test
#good = []
#best = min([m.distance for m in matches])
#for m in matches:
#  if 0.75 * m.distance < best:
#    good.append(m)

# cv.drawMatchesKnn expects list of lists as matches.
res_img = cv.drawMatches(
    img_obj,
    kp_obj,
    img_scn,
    kp_scn,
    matches,
    None,
    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)


plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()

In [ ]:
des_scn[0]

In [ ]:
# Encontramos features con ORB, pero
# si el tiempo que ganamos se nos va matcheando con fuerza bruta no nos sirve-
# => LSH

In [ ]:
algo = cv2.ORB_create()

# find the keypoints and descriptors
kp_obj, des_obj = algo.detectAndCompute(img_obj, None)
kp_scn, des_scn = algo.detectAndCompute(img_scn, None)

FLANN_INDEX_LSH = 6
index_params = dict(
     algorithm=FLANN_INDEX_LSH,
    table_number=6,  # 12
    key_size=12,     # 20
    multi_probe_level=1)  # 2
search_params = dict(checks=50)  # or pass empty dictionary

# Create FLANN based matcher object
flann = cv2.FlannBasedMatcher(index_params, search_params)

k = 2
matches = flann.knnMatch(des_obj, des_scn, k)
# Need to draw only good matches, so create a mask
matchesMask = [[0,0] for i in range(len(matches))]

# ratio test
for i, (m, n) in enumerate(matches):
   if m.distance < 0.7 * n.distance:
      matchesMask[i]=[1,0]
       
draw_params = dict(matchColor = (0,255,0),singlePointColor = (255,0,0),matchesMask = matchesMask,flags = 0)
res_img = cv2.drawMatchesKnn(img_obj,kp_obj,img_scn,kp_scn,matches,None,**draw_params)

plt.figure(figsize=(20, 16))
plt.imshow(res_img)
plt.show()

# Ejercicio

- variando los distintos métodos de detección y extracción de features SIFT, AKAZE, ORB.
- variando las imágenes de prueba suministradas o bien algunas que quieran usar tomadas con el celular.


Preguntas:
- ¿Qué método funcionan mejor en qué escenario?
- ¿Qué parámetros de los métodos podemos tunnear? ¿Cuál es el efecto logrado?
- ¿Soportan rotaciones? ¿Cambios de Escala?
- ¿Se puede correr en real time?



